## Setup PySpark

In [14]:
import findspark
findspark.init()
from pyspark.sql import SparkSession


spark = SparkSession.builder.appName("TipsData").getOrCreate()

In [15]:
tips_df = spark.read.csv('/content/tips.csv', header=True, inferSchema=True)

print("DataFrame Schema:")
tips_df.printSchema()

print("\nFirst 10 records of the DataFrame:")
tips_df.show(10)

DataFrame Schema:
root
 |-- total_bill: double (nullable = true)
 |-- tip: double (nullable = true)
 |-- gender: string (nullable = true)
 |-- smoker: string (nullable = true)
 |-- day: string (nullable = true)
 |-- time: string (nullable = true)
 |-- size: integer (nullable = true)
 |-- price_per_person: double (nullable = true)
 |-- Payer Name: string (nullable = true)
 |-- CC Number: double (nullable = true)
 |-- Payment ID: string (nullable = true)


First 10 records of the DataFrame:
+----------+----+------+------+---+------+----+----------------+------------------+----------+----------+
|total_bill| tip|gender|smoker|day|  time|size|price_per_person|        Payer Name| CC Number|Payment ID|
+----------+----+------+------+---+------+----+----------------+------------------+----------+----------+
|     16.99|1.01|Female|    No|Sun|Dinner|   2|            8.49|Christy Cunningham|3.56033E15|   Sun2959|
|     10.34|1.66|  Male|    No|Sun|Dinner|   3|            3.45|    Douglas Tucker

In [16]:
from pyspark.sql.functions import round

tips_df = tips_df.withColumn(
    "Tip_Percentage",
    round((tips_df['tip'] / tips_df['total_bill']) * 100, 2)
)

print("DataFrame with 'Tip_Percentage' column:")
tips_df.show(10)

DataFrame with 'Tip_Percentage' column:
+----------+----+------+------+---+------+----+----------------+------------------+----------+----------+--------------+
|total_bill| tip|gender|smoker|day|  time|size|price_per_person|        Payer Name| CC Number|Payment ID|Tip_Percentage|
+----------+----+------+------+---+------+----+----------------+------------------+----------+----------+--------------+
|     16.99|1.01|Female|    No|Sun|Dinner|   2|            8.49|Christy Cunningham|3.56033E15|   Sun2959|          5.94|
|     10.34|1.66|  Male|    No|Sun|Dinner|   3|            3.45|    Douglas Tucker|4.47807E15|   Sun4608|         16.05|
|     21.01| 3.5|  Male|    No|Sun|Dinner|   3|             7.0|    Travis Walters|6.01181E15|   Sun4458|         16.66|
|     23.68|3.31|  Male|    No|Sun|Dinner|   2|           11.84|  Nathaniel Harris|4.67614E15|   Sun5260|         13.98|
|     24.59|3.61|Female|    No|Sun|Dinner|   4|            6.15|      Tonya Carter|4.83273E15|   Sun2251|        

In [17]:
tips_df.createOrReplaceTempView("tips")

## Spark SQL Queries

In [18]:
# 1. Find the average total bill for each day
print("Average total bill for each day:")
avg_bill_per_day = spark.sql("SELECT day, AVG(total_bill) as Average_Total_Bill FROM tips GROUP BY day ORDER BY Average_Total_Bill DESC")
display(avg_bill_per_day)
avg_bill_per_day.show()

Average total bill for each day:


DataFrame[day: string, Average_Total_Bill: double]

+----+------------------+
| day|Average_Total_Bill|
+----+------------------+
| Sun|21.410000000000004|
| Sat|20.441379310344825|
|Thur|17.682741935483865|
| Fri|17.151578947368417|
+----+------------------+



In [19]:
# 2. Find the maximum tip given by gender
print("\nMaximum tip given by gender:")
max_tip_by_gender = spark.sql("SELECT gender, MAX(tip) as Max_Tip FROM tips GROUP BY gender")
display(max_tip_by_gender)
max_tip_by_gender.show()


Maximum tip given by gender:


DataFrame[gender: string, Max_Tip: double]

+------+-------+
|gender|Max_Tip|
+------+-------+
|Female|    6.5|
|  Male|   10.0|
+------+-------+



In [13]:
# 3. Display all records where tip percentage is greater than 20%
print("\nRecords where Tip_Percentage is greater than 20%:")
tips_gt_20_percent = spark.sql("SELECT * FROM tips WHERE Tip_Percentage > 20 ORDER BY Tip_Percentage DESC")
display(tips_gt_20_percent)
tips_gt_20_percent.show()


Records where Tip_Percentage is greater than 20%:


DataFrame[total_bill: double, tip: double, gender: string, smoker: string, day: string, time: string, size: int, price_per_person: double, Payer Name: string, CC Number: double, Payment ID: string, Tip_Percentage: double]

+----------+----+------+------+----+------+----+----------------+------------------+----------+----------+--------------+
|total_bill| tip|gender|smoker| day|  time|size|price_per_person|        Payer Name| CC Number|Payment ID|Tip_Percentage|
+----------+----+------+------+----+------+----+----------------+------------------+----------+----------+--------------+
|      7.25|5.15|  Male|   Yes| Sun|Dinner|   2|            3.62|       Larry White|3.04326E13|   Sun9209|         71.03|
|       9.6| 4.0|Female|   Yes| Sun|Dinner|   2|             4.8|      Melanie Gray|4.21181E12|   Sun4598|         41.67|
|      3.07| 1.0|Female|   Yes| Sat|Dinner|   1|            3.07|     Tiffany Brock|4.35949E15|   Sat3455|         32.57|
|     11.61|3.39|  Male|    No| Sat|Dinner|   2|             5.8|      James Taylor|6.01148E15|   Sat2124|          29.2|
|     23.17| 6.5|  Male|   Yes| Sun|Dinner|   4|            5.79| Dr. Michael James| 4.7185E12|   Sun6059|         28.05|
|     14.31| 4.0|Female|

In [12]:
from pyarrow.ipc import pa
tips_gt_20_percent.write.parquet("tips_gt_20_percent.parquet")

In [ ]:
from IPython.core import application
from IPython.core.hooks import CommandChainDispatcher
a)initialize a git repository for the spark project. create .gitignore file to exclude temporary spark output folders. commit the project and push it to github repository. show all Commands
b)create a Dockerfile to execute the spark aplication. Build an image named spark-tips:v1, run the container, and verify the application output. show the docker commands used.
c)List all Docker images on the system and remove the image spark-tips:v1. show the commands